#Initialization and Loading Library

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

#Data Transformations

##Trimming the values

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

df.display()

##Normalization

In [0]:
df = (
    df.withColumn(
        "cst_marital_status",
        when(upper(col("cst_marital_status")) == "M", lit("Married"))
        .when(upper(col("cst_marital_status")) == "S", lit("Single"))
        .otherwise(lit("n/a"))
    )
    .withColumn(
        "cst_gndr",
        when(upper(col("cst_gndr")) == "M", lit("Male"))
        .when(upper(col("cst_gndr")) == "F", lit("Female"))
        .otherwise(lit("n/a"))
    )
)

df.display()


##Renaming Columns

In [0]:
columns = df.columns
print(columns)

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date",
}



In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
df.display()

    

#Write back to Silver Table

In [0]:
(
    df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.crm_customers")
)

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers